In [7]:
import os
import json
import numpy as np
import jax
import jax.numpy as jnp

from functools import partial
from jax import random, lax, vmap


# ============================================================
# Couplings / sectors
# ============================================================

def make_couplings(Lx, Ly, J=1.0, sector=(0, 0)):
    """
    sector = (tx, ty)

    tx=1 : antiperiodic (defect seam) on the LAST COLUMN of x-bonds
           i.e. the bond connecting site (i, Lx-1) -> (i, 0)
    ty=1 : antiperiodic (defect seam) on the LAST ROW of y-bonds
           i.e. the bond connecting site (Ly-1, j) -> (0, j)
    """
    tx, ty = sector

    Jx = J * jnp.ones((Ly, Lx))
    Jy = J * jnp.ones((Ly, Lx))

    if tx:
        Jx = Jx.at[:, Lx - 1].set(-J)

    if ty:
        Jy = Jy.at[Ly - 1, :].set(-J)

    return Jx, Jy


# ============================================================
# Initial spins
# ============================================================

def random_spins(key, Lx, Ly):
    return random.choice(
        key,
        jnp.array([-1, 1], dtype=jnp.int8),
        shape=(Ly, Lx)
    )


# ============================================================
# Local field / energy
# ============================================================

def local_field(spins, Jx, Jy):

    left  = jnp.roll(spins,  1, axis=1)
    right = jnp.roll(spins, -1, axis=1)

    up    = jnp.roll(spins,  1, axis=0)
    down  = jnp.roll(spins, -1, axis=0)

    Jx_left = jnp.roll(Jx, 1, axis=1)
    Jy_up   = jnp.roll(Jy, 1, axis=0)

    return (
        Jx * right
        + Jx_left * left
        + Jy * down
        + Jy_up * up
    )


def energy(spins, Jx, Jy):

    sx = jnp.roll(spins, -1, axis=1)
    sy = jnp.roll(spins, -1, axis=0)

    ex = -jnp.sum(Jx * spins * sx)
    ey = -jnp.sum(Jy * spins * sy)

    return ex + ey


def magnetization(spins):
    return jnp.mean(spins)


# ============================================================
# Sector-aware seam observables
# ============================================================

def seam_observable(spins, sector):
    """
    Measure the bond correlator across the defect seam(s).

    For a tx=1 seam, the defect bond lives between column Lx-1 and
    column 0 (the x-wrap bond). The natural observable is the
    row-averaged correlator across that bond:

        O_x = (1/Ly) sum_i  spins[i, Lx-1] * spins[i, 0]

    For a ty=1 seam the defect bond lives between row Ly-1 and row 0:

        O_y = (1/Lx) sum_j  spins[Ly-1, j] * spins[0, j]

    When both seams are present we return both components as a tuple.

    The sign convention matches the coupling array:  a ferromagnetic
    bond contributes +1 when aligned, so <O> -> +1 in the ordered
    phase with no seam and <O> -> -1 when the seam is fully satisfied
    (domain wall threads the cut).
    """
    tx, ty = sector

    ox = jnp.mean(spins[:, -1] * spins[:, 0])   # x-seam correlator
    oy = jnp.mean(spins[-1, :] * spins[0, :])   # y-seam correlator

    if tx and not ty:
        return ox
    elif ty and not tx:
        return oy
    elif tx and ty:
        return ox, oy
    else:
        # No seam: return the bulk nearest-neighbour wrap correlators
        # (these equal +1 deep in the ordered phase regardless of sector)
        return ox, oy


def seam_correlator_x(spins):
    """Raw x-boundary bond correlator (always available)."""
    return jnp.mean(spins[:, -1] * spins[:, 0])


def seam_correlator_y(spins):
    """Raw y-boundary bond correlator (always available)."""
    return jnp.mean(spins[-1, :] * spins[0, :])


# ============================================================
# Checkerboard masks
# ============================================================

def checker_masks(Lx, Ly):

    yy, xx = jnp.indices((Ly, Lx))

    black = ((xx + yy) % 2 == 0)
    white = ~black

    return black, white


# ============================================================
# Metropolis updates
# ============================================================

def sublattice_update(spins, beta, Jx, Jy, mask, key):

    h = local_field(spins, Jx, Jy)

    dE = 2.0 * spins * h

    u = random.uniform(key, shape=spins.shape)

    accept = (
        (dE <= 0)
        |
        (jnp.log(u) < -beta * dE)
    )

    flip = mask & accept

    spins = jnp.where(flip, -spins, spins)

    return spins


def sweep(spins, beta, Jx, Jy, key, masks):

    k1, k2 = random.split(key)

    black, white = masks

    spins = sublattice_update(spins, beta, Jx, Jy, black, k1)
    spins = sublattice_update(spins, beta, Jx, Jy, white, k2)

    return spins


# ============================================================
# Jitted evolution kernel
# ============================================================

@partial(jax.jit, static_argnames=("thin",))
def evolve(spins, key, beta, Jx, Jy, masks, thin):

    def body(carry, _):
        s, k = carry
        k, kk = random.split(k)
        s = sweep(s, beta, Jx, Jy, kk, masks)
        return (s, k), None

    (spins, key), _ = lax.scan(
        body,
        (spins, key),
        None,
        length=thin
    )

    return spins, key


# ============================================================
# Bit packing
# ============================================================

def pack_spins(spins):
    """spins: {-1,+1} array  ->  packed uint8 array"""
    bits = np.asarray(spins > 0, dtype=np.uint8)
    return np.packbits(bits.reshape(-1))


def unpack_spins(packed, Ly, Lx):
    """Inverse of pack_spins."""
    bits = np.unpackbits(packed)[: Ly * Lx]
    return bits.reshape(Ly, Lx).astype(np.int8) * 2 - 1


# ============================================================
# Scalar observables recorded per snapshot
# ============================================================

def measure(spins, sector):
    """
    Return a dict of scalar observables for one snapshot.
    All quantities are dimensionless.
    """
    m  = float(magnetization(spins))
    ox = float(seam_correlator_x(spins))
    oy = float(seam_correlator_y(spins))

    return dict(
        m=m,
        m2=m * m,
        m4=m ** 4,
        abm=abs(m),
        seam_x=ox,
        seam_y=oy,
    )


# ============================================================
# Streaming Monte Carlo
# ============================================================

def run_and_stream(
    outdir,
    beta,
    Lx,
    Ly,
    sector=(0, 0),
    burn=5000,
    steps=5000,
    thin=20,
    seed=0
):
    os.makedirs(outdir, exist_ok=True)

    tag = (
        f"L{Lx}"
        f"_beta{beta:.6f}"
        f"_sector{sector[0]}{sector[1]}"
    )

    filename      = os.path.join(outdir, f"{tag}.bin")
    obs_filename  = os.path.join(outdir, f"{tag}.obs.npy")
    meta_filename = os.path.join(outdir, f"{tag}.json")

    print(f"\nStarting: {tag}")

    # --------------------------------------------------------
    # Setup
    # --------------------------------------------------------

    Jx, Jy = make_couplings(Lx, Ly, sector=sector)
    masks   = checker_masks(Lx, Ly)

    key       = random.PRNGKey(seed)
    key, k0   = random.split(key)
    spins     = random_spins(k0, Lx, Ly)

    # --------------------------------------------------------
    # Burn in
    # --------------------------------------------------------

    print("Burn-in...")

    spins, key = evolve(spins, key, beta, Jx, Jy, masks, burn)

    # --------------------------------------------------------
    # Metadata
    # --------------------------------------------------------

    metadata = {
        "Lx":     Lx,
        "Ly":     Ly,
        "beta":   float(beta),
        "sector": list(sector),
        "burn":   burn,
        "steps":  steps,
        "thin":   thin,
        "dtype":  "packbits(uint8)",
        "obs_keys": ["m", "m2", "m4", "abm", "seam_x", "seam_y"],
    }

    with open(meta_filename, "w") as f:
        json.dump(metadata, f, indent=2)

    # --------------------------------------------------------
    # Streaming write
    # --------------------------------------------------------

    print("Sampling...")

    obs_rows = []

    with open(filename, "wb") as f:

        for n in range(steps):

            spins, key = evolve(spins, key, beta, Jx, Jy, masks, thin)

            packed = pack_spins(spins)
            packed.tofile(f)

            obs = measure(spins, sector)
            obs_rows.append(list(obs.values()))

            if n % 500 == 0:
                print(f"  step {n}/{steps}", end="\r")

    # Save observable time-series as (steps, n_obs) float32 array
    np.save(obs_filename, np.array(obs_rows, dtype=np.float32))

    print(f"\nFinished: {tag}")


# ============================================================
# Production scans
# ============================================================

BETA_C = 0.44068679350977147

sizes   = [(4, 4)]
sectors = [(0, 0), (1, 0), (0, 1), (1, 1)]

critical_betas = np.array([BETA_C]) #np.linspace(0.4, 0.441, 10)


# ============================================================
# Critical FSS dataset
# ============================================================

for sector in sectors:
    for Lx, Ly in sizes:
        for beta in critical_betas:
            run_and_stream(
                outdir="tiny_ising_3x3_critical",
                beta=beta,
                Lx=Lx,
                Ly=Ly,
                sector=sector,
                burn=5000,       # keep full trajectory
                steps=10000,
                thin=10,
                seed=5678,
            )



Starting: L4_beta0.440687_sector00
Burn-in...
Sampling...
  step 9500/10000
Finished: L4_beta0.440687_sector00

Starting: L4_beta0.440687_sector10
Burn-in...
Sampling...
  step 9500/10000
Finished: L4_beta0.440687_sector10

Starting: L4_beta0.440687_sector01
Burn-in...
Sampling...
  step 9500/10000
Finished: L4_beta0.440687_sector01

Starting: L4_beta0.440687_sector11
Burn-in...
Sampling...
  step 9500/10000
Finished: L4_beta0.440687_sector11
